# Graph Theory for Visual Learners

In [ ]:
import time
from dataclasses import dataclass, field


@dataclass
class Canvas:
    """Global canvas styling shared by notebook renderers."""

    background_color: str = "#ffffff"


@dataclass
class Camera:
    """2D camera state with smooth interpolation toward target centers."""

    center: tuple[float, float, float] = (0.0, 0.0, 0.0)
    height: float = 480.0
    _start_center: tuple[float, float, float] = field(default=(0.0, 0.0, 0.0), init=False, repr=False)
    _target_center: tuple[float, float, float] = field(default=(0.0, 0.0, 0.0), init=False, repr=False)
    _start_height: float = field(default=480.0, init=False, repr=False)
    _target_height: float = field(default=480.0, init=False, repr=False)
    _start_time: float = field(default=0.0, init=False, repr=False)
    _duration: float = field(default=0.0, init=False, repr=False)
    _moving: bool = field(default=False, init=False, repr=False)

    @staticmethod
    def _zoom_from_z(z: float) -> float:
        """Map z-distance to visible world height with a floor for readability."""
        # In this 2D visualizer, z acts as camera distance, which maps to visible world height.
        return max(60.0, abs(float(z)))

    def _sample(self, now: float | None = None):
        """Advance camera interpolation and return current `(x, y, z)` center."""
        if not self._moving:
            return self.center

        if now is None:
            now = time.perf_counter()

        progress = (now - self._start_time) / self._duration
        if progress >= 1.0:
            self.center = self._target_center
            self.height = self._target_height
            self._moving = False
            return self.center

        progress = max(0.0, min(1.0, progress))
        eased = progress * progress * (3.0 - 2.0 * progress)

        sx, sy, sz = self._start_center
        tx, ty, tz = self._target_center
        self.center = (
            sx + (tx - sx) * eased,
            sy + (ty - sy) * eased,
            sz + (tz - sz) * eased,
        )
        self.height = self._start_height + (self._target_height - self._start_height) * eased
        return self.center

    def move_to(self, x: float, y: float, z: float, duration: float = 1.2):
        """Start a camera move toward `(x, y, z)` over `duration` seconds and return `self`."""
        current = self._sample()
        self.center = current
        self._start_center = current
        self._target_center = (float(x), float(y), float(z))
        self._start_height = float(self.height)
        self._target_height = self._zoom_from_z(z)
        self._start_time = time.perf_counter()
        self._duration = max(0.01, float(duration))
        self._moving = True
        return self

    def state(self):
        """Return serializable camera state as `{"center": [x, y, z], "height": h, "moving": bool}`."""
        cx, cy, cz = self._sample()
        return {
            "center": [cx, cy, cz],
            "height": float(self.height),
            "moving": bool(self._moving),
        }


canvas = Canvas()

camera = Camera()

import math
import time
from dataclasses import dataclass, field
from typing import Any, Callable


_ANIMATION_COLOR_ALIASES = {
    "black": "#000000",
    "white": "#ffffff",
    "gray": "#808080",
    "grey": "#808080",
    "red": "#ff0000",
    "green": "#008000",
    "blue": "#0000ff",
    "yellow": "#ffff00",
    "orange": "#ffa500",
    "purple": "#800080",
    "pink": "#ffc0cb",
    "teal": "#008080",
    "cyan": "#00ffff",
    "amber": "#ffbf00",
}


def _finite_float(value: Any, *, name: str) -> float:
    """Coerce a value to a finite float or raise `ValueError` naming the offending field."""
    try:
        numeric = float(value)
    except Exception as exc:
        raise ValueError(f"{name} must be a finite number") from exc
    if not math.isfinite(numeric):
        raise ValueError(f"{name} must be a finite number")
    return numeric


def _normalize_duration(duration: Any) -> float:
    """Normalize animation duration to a finite float with a minimum of `0.01` seconds."""
    return max(0.01, _finite_float(duration, name="duration"))


def _smoothstep(progress: float) -> float:
    """Clamp progress to `[0, 1]` and apply smoothstep easing."""
    clamped = max(0.0, min(1.0, progress))
    return clamped * clamped * (3.0 - 2.0 * clamped)


def _normalize_hex_color(raw_color: Any, *, name: str = "color") -> str:
    """Normalize color-like input to lowercase `#rrggbb`, validating aliases and hex format."""
    if raw_color is None:
        raise ValueError(f"{name} cannot be None")

    try:
        text = str(raw_color).strip().lower()
    except Exception as exc:
        raise ValueError(f"{name} must be a valid hex color string") from exc

    if not text:
        raise ValueError(f"{name} cannot be empty")

    text = _ANIMATION_COLOR_ALIASES.get(text, text)
    if not text.startswith("#"):
        raise ValueError(f"{name} must be a valid hex color string")

    hex_part = text[1:]
    if len(hex_part) == 3:
        if not all(ch in "0123456789abcdef" for ch in hex_part):
            raise ValueError(f"{name} must be a valid hex color string")
        hex_part = "".join(ch * 2 for ch in hex_part)
    elif len(hex_part) == 6:
        if not all(ch in "0123456789abcdef" for ch in hex_part):
            raise ValueError(f"{name} must be a valid hex color string")
    else:
        raise ValueError(f"{name} must be a valid hex color string")

    return f"#{hex_part}"


def _hex_to_rgb(hex_color: str) -> tuple[int, int, int]:
    """Convert a hex color string to an `(r, g, b)` tuple of integers."""
    normalized = _normalize_hex_color(hex_color)
    raw = normalized[1:]
    return (
        int(raw[0:2], 16),
        int(raw[2:4], 16),
        int(raw[4:6], 16),
    )


def _rgb_to_hex(rgb: tuple[float, float, float]) -> str:
    """Convert RGB channel values to a clamped lowercase `#rrggbb` string."""
    red, green, blue = rgb
    r = max(0, min(255, int(round(red))))
    g = max(0, min(255, int(round(green))))
    b = max(0, min(255, int(round(blue))))
    return f"#{r:02x}{g:02x}{b:02x}"


def _lerp_float(start: float, end: float, t: float) -> float:
    """Linearly interpolate between scalar values."""
    return start + (end - start) * t


def _lerp_color(start: str, end: str, t: float) -> str:
    """Linearly interpolate between two hex colors in RGB space."""
    start_rgb = _hex_to_rgb(start)
    end_rgb = _hex_to_rgb(end)
    return _rgb_to_hex(
        (
            _lerp_float(start_rgb[0], end_rgb[0], t),
            _lerp_float(start_rgb[1], end_rgb[1], t),
            _lerp_float(start_rgb[2], end_rgb[2], t),
        )
    )


@dataclass
class _PropertyAnimation:
    """Generic interpolation record for a single animated metadata property."""

    start: Any
    end: Any
    start_time: float
    duration: float
    interpolate: Callable[[Any, Any, float], Any]
    apply: Callable[[Any], None]

    def sample(self, now: float) -> bool:
        """Apply interpolated value at `now`; return `True` once animation reaches its target."""
        progress = (now - self.start_time) / self.duration
        if progress >= 1.0:
            self.apply(self.end)
            return True
        if progress <= 0.0:
            self.apply(self.start)
            return False

        eased = _smoothstep(progress)
        self.apply(self.interpolate(self.start, self.end, eased))
        return False


class _StyleAnimationMixin:
    """Shared helpers for scalar/color metadata animations on nodes and edges."""

    _animations: dict[str, _PropertyAnimation]

    def _animation_metadata(self):
        """Return the metadata object that owns animated style fields."""
        return self.metadata

    def _read_scalar_metadata(self, attr_name: str, *, min_value: float = 0.0) -> float:
        """Read and sanitize a numeric metadata field, persisting the normalized value."""
        metadata = self._animation_metadata()
        raw = getattr(metadata, attr_name, min_value)
        try:
            numeric = float(raw)
        except Exception:
            numeric = min_value
        if not math.isfinite(numeric):
            numeric = min_value
        numeric = max(min_value, numeric)
        setattr(metadata, attr_name, numeric)
        return numeric

    def _read_color_metadata(self, attr_name: str, *, fallback: str) -> str:
        """Read and normalize a color metadata field, falling back to a valid default."""
        metadata = self._animation_metadata()
        raw = getattr(metadata, attr_name, fallback)
        try:
            normalized = _normalize_hex_color(raw, name=attr_name)
        except ValueError:
            normalized = _normalize_hex_color(fallback, name=attr_name)
        setattr(metadata, attr_name, normalized)
        return normalized

    def _set_scalar_metadata(self, attr_name: str, value: float, *, min_value: float = 0.0) -> float:
        """Validate and assign a numeric metadata field; return stored value."""
        numeric = _finite_float(value, name=attr_name)
        if numeric < min_value:
            raise ValueError(f"{attr_name} must be >= {min_value}")
        metadata = self._animation_metadata()
        setattr(metadata, attr_name, numeric)
        return numeric

    def _set_color_metadata(self, attr_name: str, value) -> str:
        """Validate and assign a color metadata field; return normalized hex color."""
        normalized = _normalize_hex_color(value, name=attr_name)
        metadata = self._animation_metadata()
        setattr(metadata, attr_name, normalized)
        return normalized

    def _start_scalar_animation(
        self,
        attr_name: str,
        target,
        duration: float,
        *,
        now: float | None = None,
        min_value: float = 0.0,
    ):
        """Start/restart scalar animation for `attr_name`; stores a `_PropertyAnimation` in `_animations`."""
        if now is None:
            now = time.perf_counter()

        current = self._read_scalar_metadata(attr_name, min_value=min_value)
        target_value = _finite_float(target, name=attr_name)
        if target_value < min_value:
            raise ValueError(f"{attr_name} must be >= {min_value}")

        self._animations.pop(attr_name, None)
        if math.isclose(current, target_value, rel_tol=0.0, abs_tol=1e-9):
            self._set_scalar_metadata(attr_name, target_value, min_value=min_value)
            return self

        self._animations[attr_name] = _PropertyAnimation(
            start=current,
            end=target_value,
            start_time=now,
            duration=_normalize_duration(duration),
            interpolate=_lerp_float,
            apply=lambda value, key=attr_name, lower=min_value: self._set_scalar_metadata(key, value, min_value=lower),
        )
        return self

    def _start_color_animation(self, attr_name: str, target, duration: float, *, now: float | None = None):
        """Start/restart color animation for `attr_name` toward normalized `target`."""
        if now is None:
            now = time.perf_counter()

        target_value = _normalize_hex_color(target, name=attr_name)
        current = self._read_color_metadata(attr_name, fallback=target_value)

        self._animations.pop(attr_name, None)
        if current == target_value:
            self._set_color_metadata(attr_name, target_value)
            return self

        self._animations[attr_name] = _PropertyAnimation(
            start=current,
            end=target_value,
            start_time=now,
            duration=_normalize_duration(duration),
            interpolate=_lerp_color,
            apply=lambda value, key=attr_name: self._set_color_metadata(key, value),
        )
        return self

    def _sample_style_animations(self, now: float) -> bool:
        """Advance all active style animations and return `True` while any remain active."""
        if not self._animations:
            return False

        # Iterate over a snapshot so completed animations can be removed mid-loop.
        for attr_name, animation in list(self._animations.items()):
            completed = animation.sample(now)
            if completed:
                self._animations.pop(attr_name, None)

        return bool(self._animations)

    def _style_moving(self) -> bool:
        """Return whether any style animations are still in-flight."""
        return bool(self._animations)


@dataclass
class NodeMeta:
    """Per-node style settings consumed by renderers and style animations."""

    fill_color: str = "#e8c547"
    stroke_color: str = "#1f2937"
    stroke_width: float = 3.0
    diameter: float = 50.0
    label_size: float = 30.0
    label_color: str = "#111827"
    display_label: bool = True


@dataclass
class Node(_StyleAnimationMixin):
    """Graph node with position, style metadata, and optional motion animations."""

    id: str
    label: str | None = None
    metadata: NodeMeta = field(default_factory=NodeMeta)
    pos: tuple[float, float] | None = None
    _start_pos: tuple[float, float] = field(default=(0.0, 0.0), init=False, repr=False)
    _target_pos: tuple[float, float] = field(default=(0.0, 0.0), init=False, repr=False)
    _start_time: float = field(default=0.0, init=False, repr=False)
    _duration: float = field(default=0.0, init=False, repr=False)
    _pos_moving: bool = field(default=False, init=False, repr=False)
    _moving: bool = field(default=False, init=False, repr=False)
    _wiggle_start_pos: tuple[float, float] = field(default=(0.0, 0.0), init=False, repr=False)
    _wiggle_start_time: float = field(default=0.0, init=False, repr=False)
    _wiggle_duration: float = field(default=0.0, init=False, repr=False)
    _wiggle_speed: float = field(default=0.0, init=False, repr=False)
    _wiggle_temperature: float = field(default=0.0, init=False, repr=False)
    _wiggle_phase_x: float = field(default=0.0, init=False, repr=False)
    _wiggle_phase_y: float = field(default=0.0, init=False, repr=False)
    _wiggle_freq_x: float = field(default=1.0, init=False, repr=False)
    _wiggle_freq_y: float = field(default=1.0, init=False, repr=False)
    _wiggle_active: bool = field(default=False, init=False, repr=False)
    _animations: dict[str, _PropertyAnimation] = field(default_factory=dict, init=False, repr=False)

    def __post_init__(self):
        """Normalize user-provided identifiers/labels and enforce non-empty string IDs."""
        node_id = str(self.id)
        if not node_id.strip():
            raise ValueError("node id cannot be empty")
        self.id = node_id
        if self.label is None:
            self.label = self.id
        else:
            self.label = str(self.label)

    @staticmethod
    def _parse_pos(raw_pos):
        """Parse `(x, y)` from tuple/list/dict inputs; return `None` when invalid."""
        if isinstance(raw_pos, (list, tuple)) and len(raw_pos) >= 2:
            try:
                x = float(raw_pos[0])
                y = float(raw_pos[1])
            except Exception:
                return None
            if math.isfinite(x) and math.isfinite(y):
                return (x, y)
            return None

        if isinstance(raw_pos, dict):
            try:
                x = float(raw_pos.get("x"))
                y = float(raw_pos.get("y"))
            except Exception:
                return None
            if math.isfinite(x) and math.isfinite(y):
                return (x, y)
            return None

        return None

    @staticmethod
    def _seed_unit(text: str, salt: int) -> float:
        """Generate deterministic pseudo-random unit value from node identity and salt."""
        value = 0
        for index, char in enumerate(f"{text}:{salt}"):
            value = (value * 131 + (index + salt + 1) * ord(char)) % 104729
        return value / 104729.0

    def _start_wiggle(self, speed: float, duration: float, temperature: float, *, now: float | None = None):
        """Configure deterministic wiggle animation; returns `self` for chaining."""
        speed_value = _finite_float(speed, name="speed")
        if speed_value <= 0.0:
            raise ValueError("speed must be > 0")
        duration_value = _normalize_duration(duration)
        temperature_value = _finite_float(temperature, name="temperature")
        if temperature_value < 0.0 or temperature_value > 1.0:
            raise ValueError("temperature must be between 0 and 1 inclusive")

        if now is None:
            now = time.perf_counter()

        current = self._sample(now=now)
        if current is None:
            current = self._parse_pos(self.pos)
        if current is None:
            current = self._target_pos
            self.pos = current

        node_identity = str(self.id)
        # Tie phase/frequency to node id so each node has stable motion characteristics.
        self._wiggle_phase_x = 2.0 * math.pi * self._seed_unit(node_identity, 11)
        self._wiggle_phase_y = 2.0 * math.pi * self._seed_unit(node_identity, 29)
        self._wiggle_freq_x = 0.8 + (1.6 * self._seed_unit(node_identity, 47))
        self._wiggle_freq_y = 0.8 + (1.6 * self._seed_unit(node_identity, 83))
        self._wiggle_start_pos = current
        self._wiggle_start_time = now
        self._wiggle_duration = duration_value
        self._wiggle_speed = speed_value
        self._wiggle_temperature = temperature_value
        self._wiggle_active = temperature_value > 0.0
        self._moving = bool(self._pos_moving or self._style_moving() or self._wiggle_active)
        return self

    def _sample_wiggle(self, now: float, current_pos: tuple[float, float] | None):
        """Sample wiggle offset for `current_pos`; returns `((x, y) | None, is_active)`."""
        if not self._wiggle_active:
            return current_pos, False
        if current_pos is None:
            return current_pos, False

        progress = (now - self._wiggle_start_time) / self._wiggle_duration
        if progress >= 1.0:
            self._wiggle_active = False
            self.pos = self._wiggle_start_pos
            return self._wiggle_start_pos, False

        clamped_progress = max(0.0, min(1.0, progress))
        # Envelope keeps wiggle strongest mid-animation and smooth at boundaries.
        envelope = math.sin(math.pi * clamped_progress) ** 2
        phase_time = (now - self._wiggle_start_time) * self._wiggle_speed

        chaos = self._wiggle_temperature
        detail_mix = 0.25 + (0.55 * chaos)
        amplitude = (5.0 + (15.0 * chaos)) * chaos

        wave_x = math.sin((phase_time * self._wiggle_freq_x) + self._wiggle_phase_x)
        wave_x += detail_mix * math.sin((phase_time * (self._wiggle_freq_x * (1.7 + chaos))) + (self._wiggle_phase_y * 0.7))
        wave_y = math.cos((phase_time * self._wiggle_freq_y) + self._wiggle_phase_y)
        wave_y += detail_mix * math.sin((phase_time * (self._wiggle_freq_y * (2.1 + chaos))) + (self._wiggle_phase_x * 1.3))

        offset_scale = amplitude * envelope
        return (
            self._wiggle_start_pos[0] + (wave_x * offset_scale),
            self._wiggle_start_pos[1] + (wave_y * offset_scale),
        ), True

    def _sample(self, now: float | None = None):
        """Advance position/style/wiggle animations and return current `(x, y)` or `None`."""
        if now is None:
            now = time.perf_counter()

        current_pos = self._parse_pos(self.pos)
        if self._pos_moving:
            progress = (now - self._start_time) / self._duration
            if progress >= 1.0:
                current_pos = self._target_pos
                self.pos = current_pos
                self._pos_moving = False
            elif progress <= 0.0:
                current_pos = self._start_pos
                self.pos = current_pos
            else:
                eased = _smoothstep(progress)
                sx, sy = self._start_pos
                tx, ty = self._target_pos
                current_pos = (
                    sx + (tx - sx) * eased,
                    sy + (ty - sy) * eased,
                )
                self.pos = current_pos

        current_pos, wiggle_moving = self._sample_wiggle(now, current_pos)
        style_moving = self._sample_style_animations(now)
        self._moving = bool(self._pos_moving or style_moving or wiggle_moving)
        return current_pos

    def move_to(self, x: float, y: float, duration: float = 1.2):
        """Animate node position toward `(x, y)` over `duration` seconds; returns `self`."""
        target = (
            _finite_float(x, name="x"),
            _finite_float(y, name="y"),
        )

        now = time.perf_counter()
        current = self._sample(now=now)
        self._wiggle_active = False
        if current is None:
            current = self._parse_pos(self.pos)
        if current is None:
            current = self._target_pos
            self.pos = current

        self._start_pos = current
        self._target_pos = target
        self._start_time = now
        self._duration = _normalize_duration(duration)
        self._pos_moving = self._start_pos != self._target_pos
        if not self._pos_moving:
            self.pos = self._target_pos

        self._moving = bool(self._pos_moving or self._style_moving() or self._wiggle_active)
        return self

    def change_diameter(self, to, duration: float = 1.2):
        """Animate `metadata.diameter` to `to` over `duration` seconds; returns `self`."""
        now = time.perf_counter()
        self._sample(now=now)
        self._start_scalar_animation("diameter", to, duration, now=now, min_value=0.0)
        self._moving = bool(self._pos_moving or self._style_moving() or self._wiggle_active)
        return self

    def change_fill_color(self, to, duration: float = 1.2):
        """Animate `metadata.fill_color` to `to` over `duration` seconds; returns `self`."""
        now = time.perf_counter()
        self._sample(now=now)
        self._start_color_animation("fill_color", to, duration, now=now)
        self._moving = bool(self._pos_moving or self._style_moving() or self._wiggle_active)
        return self

    def change_stroke_color(self, to, duration: float = 1.2):
        """Animate `metadata.stroke_color` to `to` over `duration` seconds; returns `self`."""
        now = time.perf_counter()
        self._sample(now=now)
        self._start_color_animation("stroke_color", to, duration, now=now)
        self._moving = bool(self._pos_moving or self._style_moving() or self._wiggle_active)
        return self

    def change_stroke_width(self, to, duration: float = 1.2):
        """Animate `metadata.stroke_width` to `to` over `duration` seconds; returns `self`."""
        now = time.perf_counter()
        self._sample(now=now)
        self._start_scalar_animation("stroke_width", to, duration, now=now, min_value=0.0)
        self._moving = bool(self._pos_moving or self._style_moving() or self._wiggle_active)
        return self

    def change_label_size(self, to, duration: float = 1.2):
        """Animate `metadata.label_size` to `to` over `duration` seconds; returns `self`."""
        now = time.perf_counter()
        self._sample(now=now)
        self._start_scalar_animation("label_size", to, duration, now=now, min_value=0.0)
        self._moving = bool(self._pos_moving or self._style_moving() or self._wiggle_active)
        return self

    def scale(self, factor, duration: float = 1.2):
        """Scale diameter/stroke/label sizes together by `factor` over `duration`; returns `self`."""
        factor_value = _finite_float(factor, name="factor")
        if factor_value <= 0.0:
            raise ValueError("factor must be > 0")

        now = time.perf_counter()
        self._sample(now=now)
        duration_value = _normalize_duration(duration)

        current_diameter = self._read_scalar_metadata("diameter", min_value=0.0)
        current_stroke_width = self._read_scalar_metadata("stroke_width", min_value=0.0)
        current_label_size = self._read_scalar_metadata("label_size", min_value=0.0)

        self._start_scalar_animation("diameter", current_diameter * factor_value, duration_value, now=now, min_value=0.0)
        self._start_scalar_animation("stroke_width", current_stroke_width * factor_value, duration_value, now=now, min_value=0.0)
        self._start_scalar_animation("label_size", current_label_size * factor_value, duration_value, now=now, min_value=0.0)
        self._moving = bool(self._pos_moving or self._style_moving() or self._wiggle_active)
        return self

    def state(self):
        """Return serializable node state dict with position, style fields, and `moving` flag."""
        now = time.perf_counter()
        pos = self._sample(now=now)
        return {
            "pos": [pos[0], pos[1]] if pos is not None else None,
            "fill_color": self._read_color_metadata("fill_color", fallback="#e8c547"),
            "stroke_color": self._read_color_metadata("stroke_color", fallback="#1f2937"),
            "stroke_width": self._read_scalar_metadata("stroke_width", min_value=0.0),
            "diameter": self._read_scalar_metadata("diameter", min_value=0.0),
            "label_size": self._read_scalar_metadata("label_size", min_value=0.0),
            "label_color": self._read_color_metadata("label_color", fallback="#111827"),
            "moving": bool(self._moving),
        }


@dataclass
class EdgeMeta:
    """Per-edge style settings used by renderers and style animations."""

    color: str = "#8a8a8a"
    width: float = 4.0
    display_label: bool = False


@dataclass
class Edge(_StyleAnimationMixin):
    """Directed connection between two nodes with animated style metadata."""

    frm: Node
    to: Node
    label: str = ""
    metadata: EdgeMeta = field(default_factory=EdgeMeta)
    _moving: bool = field(default=False, init=False, repr=False)
    _animations: dict[str, _PropertyAnimation] = field(default_factory=dict, init=False, repr=False)

    def _sample(self, now: float | None = None):
        """Advance style animation state and return `self`."""
        if now is None:
            now = time.perf_counter()
        self._moving = bool(self._sample_style_animations(now))
        return self

    def change_width(self, to, duration: float = 1.2):
        """Animate `metadata.width` to `to` over `duration` seconds; returns `self`."""
        now = time.perf_counter()
        self._sample(now=now)
        self._start_scalar_animation("width", to, duration, now=now, min_value=0.0)
        self._moving = bool(self._style_moving())
        return self

    def change_color(self, to, duration: float = 1.2):
        """Animate `metadata.color` to `to` over `duration` seconds; returns `self`."""
        now = time.perf_counter()
        self._sample(now=now)
        self._start_color_animation("color", to, duration, now=now)
        self._moving = bool(self._style_moving())
        return self

    def state(self):
        """Return serializable edge style state with color, width, and moving flag."""
        now = time.perf_counter()
        self._sample(now=now)
        return {
            "color": self._read_color_metadata("color", fallback="#8a8a8a"),
            "width": self._read_scalar_metadata("width", min_value=0.0),
            "moving": bool(self._moving),
        }


@dataclass
class GraphMeta:
    """Top-level graph metadata for labels/layout hints in renderers."""

    label: str = ""
    layout: str = "force"
    display_label: bool = False


class Graph:
    """Container for nodes/edges with coercion helpers and graph-wide animation controls."""

    def __init__(self, nodes, edges, metadata=None):
        """Build a graph from node/edge iterables, coercing inputs through add helpers."""
        self.metadata = metadata if isinstance(metadata, GraphMeta) else GraphMeta()
        self.nodes = []
        self.edges = []

        for node in nodes:
            self.add_node(node)

        for edge in edges:
            self.add_edge(edge)

    def _find_node_by_id(self, node_id):
        """Return the first node whose stringified id matches `node_id`, else `None`."""
        text = str(node_id)
        for candidate in self.nodes:
            if str(candidate.id) == text:
                return candidate
        return None

    def _coerce_node(self, node):
        """Coerce shorthand node inputs into a `Node` instance without inserting it."""
        if isinstance(node, Node):
            return node
        if node is None:
            raise ValueError("node cannot be None")
        if isinstance(node, tuple):
            if len(node) != 2:
                raise ValueError("node tuple shorthand must be (id, label)")
            node_id, node_label = node
            return Node(id=str(node_id), label=None if node_label is None else str(node_label))
        return Node(id=str(node))

    def _ensure_node(self, node):
        """Return canonical node object in the graph, matching by identity first then id."""
        candidate = self._coerce_node(node)

        # Preserve explicit object identity when callers reuse the same Node instance.
        for existing in self.nodes:
            if existing is candidate:
                return existing

        # Fall back to id-based matching so shorthand inputs reuse existing logical nodes.
        by_id = self._find_node_by_id(candidate.id)
        if by_id is not None:
            return by_id

        self.nodes.append(candidate)
        return candidate

    def _coerce_edge(self, edge, anchor_node=None):
        """Coerce edge shorthands into an `Edge`, reusing canonical endpoint node objects."""
        if isinstance(edge, Edge):
            frm = self._ensure_node(edge.frm)
            to = self._ensure_node(edge.to)
            if frm is edge.frm and to is edge.to:
                return edge
            return Edge(frm=frm, to=to, label=edge.label, metadata=edge.metadata)

        if isinstance(edge, tuple):
            edge_values = list(edge)
        elif isinstance(edge, list):
            edge_values = edge
        elif anchor_node is not None:
            edge_values = [anchor_node, edge]
        else:
            raise TypeError("edge must be an Edge or an endpoint sequence")

        if len(edge_values) == 0:
            raise ValueError("edge endpoint sequence cannot be empty")

        if len(edge_values) == 1:
            if anchor_node is None:
                raise ValueError("single-endpoint edge requires an anchor node")
            # Single endpoint shorthand means "anchor_node -> endpoint".
            frm = anchor_node
            to = self._ensure_node(edge_values[0])
            return Edge(frm=frm, to=to)

        frm = self._ensure_node(edge_values[0])
        to = self._ensure_node(edge_values[1])
        label = ""
        metadata = None

        if len(edge_values) >= 3 and edge_values[2] is not None:
            label = str(edge_values[2])
        # Fourth slot is reserved for EdgeMeta; other extras are intentionally ignored.
        if len(edge_values) >= 4 and isinstance(edge_values[3], EdgeMeta):
            metadata = edge_values[3]

        if metadata is None:
            return Edge(frm=frm, to=to, label=label)
        return Edge(frm=frm, to=to, label=label, metadata=metadata)

    def add_node(self, node, edges=None):
        """Insert/reuse a node and optionally coerce/add attached edge shorthand definitions."""
        node_obj = self._ensure_node(node)

        if edges is None:
            return node_obj

        if isinstance(edges, tuple):
            edge_inputs = [edges]
        elif isinstance(edges, (list, set, frozenset)):
            edge_inputs = list(edges)
        else:
            edge_inputs = [edges]

        for edge_input in edge_inputs:
            if edge_input is None:
                continue
            if isinstance(edge_input, Edge):
                self.add_edge(edge_input)
                continue
            if isinstance(edge_input, tuple):
                if len(edge_input) == 0:
                    continue
                if len(edge_input) == 1:
                    self.add_edge((node_obj, edge_input[0]))
                else:
                    self.add_edge(edge_input)
                continue
            if isinstance(edge_input, list):
                if len(edge_input) == 0:
                    continue
                if len(edge_input) == 1:
                    self.add_edge((node_obj, edge_input[0]))
                else:
                    self.add_edge(edge_input)
                continue
            self.add_edge((node_obj, edge_input))

        return node_obj

    def add_edge(self, edge):
        """Insert an edge after coercion and return the canonical `Edge` object."""
        edge_obj = self._coerce_edge(edge)
        self.edges.append(edge_obj)
        return edge_obj

    def wiggle(self, speed: float, duration: float, temperature: float):
        """Start synchronized wiggle animation for all nodes; returns `self`."""
        speed_value = _finite_float(speed, name="speed")
        if speed_value <= 0.0:
            raise ValueError("speed must be > 0")
        duration_value = _normalize_duration(duration)
        temperature_value = _finite_float(temperature, name="temperature")
        if temperature_value < 0.0 or temperature_value > 1.0:
            raise ValueError("temperature must be between 0 and 1 inclusive")

        now = time.perf_counter()
        for node in self.nodes:
            node._start_wiggle(speed=speed_value, duration=duration_value, temperature=temperature_value, now=now)
        return self

    def __str__(self):
        """Return a readable multiline summary of nodes and directed edges."""
        node_descriptions = ", ".join(
            node.id if node.label == node.id else f"{node.id} ({node.label})"
            for node in self.nodes
        ) or "(none)"
        edge_lines = [f"  - {edge.frm.id} -> {edge.to.id}" for edge in self.edges]
        edges_block = "\n".join(edge_lines) if edge_lines else "  - (none)"
        return f"Graph:\n  Nodes: {node_descriptions}\n  Edges:\n{edges_block}"

    def __repr__(self):
        """Mirror `__str__` for debugger-friendly display."""
        return self.__str__()


In [ ]:
graph_1 = Graph(
    nodes=["a", "b", "c", "d", ("e", "$ e_1 $")],
    edges=[("a", "b"), ("b", "c"), ("c", "d"), ("d", "e"), ("e", "a"),
           ("a", "c"), ("a", "d")]
)

In [ ]:
graph_1.wiggle(speed=1, duration=10, temperature=0.5)

In [ ]:
graph_1.nodes[0].change_fill_color(to="#ffffff", duration=5)

In [ ]:
graph_1.nodes

In [ ]:
graph_2 = Graph(
    nodes=["a", "b"],
    edges=[("a", "b"), ("b", "c")]
)